# Biological train/validation/test split (before annotation)

Split `Data/DeathScope_YOLOv8_DataSet` **before cropping, annotation, and augmentation** so related images cannot leak between partitions. Run this notebook **before `annotate_cell_images.ipynb`**. Only Python's standard library is required. Run from the repository root.

**Run order:** biological split → annotate each split separately → augment training data only. The inputs here are raw phase/Green images; annotations are generated afterward.

The expected source is `Batch*/<DeathType>/<CellType>_phase/` with matching `<CellType>_Green/` files. For `VID795_H11_1_00d18h00m.png`, we interpret `VID795` as acquisition/video, `H11` as well, `1` as field, and the final token as time. **Confirm that wells represent your biological units.** If independent experiments are the biological units, select `batch` instead. Well-level splitting does not measure generalization to unseen batches.

- `well` (default): group by batch + video + well, keeping every field, time point, and both channels together. Randomize wells within each death-type/cell-type stratum using a fixed seed.
- `batch`: keep each entire batch together. With only two batches, use two partitions, e.g. `train=0.5, val=0.5, test=0`. Three nonempty partitions require at least three batches.

Ratios apply to **groups**, not images. Small numbers of wells and different well sizes mean actual image ratios will differ. Every requested partition receives at least one group in each stratum; insufficient groups raise an error.

The source is preserved. The final cell copies paired raw images to a fresh output folder, writes a per-pair manifest and an unmatched-file report, and records configuration. No YOLO labels exist at this raw-data stage.


In [ ]:
from pathlib import Path
from collections import defaultdict, Counter
import csv
import json
import math
import random
import re
import shutil

SOURCE_ROOT = Path("Data/DeathScope_YOLOv8_DataSet")
OUTPUT_ROOT = Path("Data/DeathScope_biological_split")
GROUP_LEVEL = "well"  # "well" or "batch"
SPLIT_RATIOS = {"train": 0.8, "val": 0.1, "test": 0.1}
SEED = 99
# "skip": exclude unmatched files and record them; "error": require complete pairing.
# The supplied dataset contains unmatched files; inspect the report before training.
MISSING_PAIR_POLICY = "skip"
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
FILENAME_PATTERN = re.compile(
    r"(?P<video>VID\d+)_(?P<well>[A-Za-z]+\d+)_(?P<field>\d+)_"
    r"(?P<time>\d+d\d+h\d+m)", re.IGNORECASE
)


## Discover paired images and biological units

Pair channels by exact filename. Unexpected filenames fail rather than guessing a biological identity. Missing channels are listed below and saved in `unmatched_files.csv` on export. Set `MISSING_PAIR_POLICY="error"` to stop on any missing partner.


In [ ]:
def discover_pairs(source_root, missing_policy="skip"):
    source_root = Path(source_root)
    if missing_policy not in {"skip", "error"}:
        raise ValueError("missing_policy must be 'skip' or 'error'")
    if not source_root.is_dir():
        raise FileNotFoundError(source_root)
    pairs, unmatched = [], []
    channel_dirs = sorted(p for p in source_root.glob("*/*/*") if p.is_dir())
    phase_dirs = [p for p in channel_dirs if p.name.lower().endswith("_phase")]
    # Also catch Green-only directories rather than silently losing their files.
    for green in (p for p in channel_dirs if p.name.endswith("_Green")):
        cell = green.name[:-6]
        if not any(p.parent == green.parent and p.name[:-6] == cell for p in phase_dirs):
            for image in sorted(green.iterdir()):
                if image.suffix.lower() in IMAGE_EXTENSIONS:
                    unmatched.append({"path": str(image.relative_to(source_root)), "reason": "missing phase channel"})
    for phase in phase_dirs:
        cell = phase.name[:-6]
        green = phase.parent / (cell + "_Green")
        phase_files = {p.name: p for p in phase.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS}
        green_files = {p.name: p for p in green.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS} if green.is_dir() else {}
        for name in sorted(phase_files.keys() | green_files.keys()):
            if name not in phase_files or name not in green_files:
                existing = phase_files.get(name, green_files.get(name))
                unmatched.append({"path": str(existing.relative_to(source_root)), "reason": "missing Green channel" if name not in green_files else "missing phase channel"})
                continue
            match = FILENAME_PATTERN.fullmatch(Path(name).stem)
            if match is None:
                raise ValueError("Unexpected raw filename; update FILENAME_PATTERN: " + name)
            batch, death = phase.relative_to(source_root).parts[:2]
            pairs.append(dict(batch=batch, death_type=death, cell_type=cell,
                              **match.groupdict(),
                              phase=str(phase_files[name].relative_to(source_root)),
                              green=str(green_files[name].relative_to(source_root))))
    if missing_policy == "error" and unmatched:
        raise ValueError("{} unmatched images; examples: {}".format(len(unmatched), unmatched[:5]))
    if not pairs:
        raise ValueError("No paired raw images found under " + str(source_root))
    return pairs, unmatched

pairs, unmatched = discover_pairs(SOURCE_ROOT, MISSING_PAIR_POLICY)
print("Paired images:", len(pairs), "| Unmatched images excluded:", len(unmatched))
for row in unmatched:
    print(row["reason"], row["path"])


## Assign complete groups and preview the split

The assignment checks that a group belongs to exactly one partition. No files are copied by this cell. Well groups appearing under multiple death/cell strata raise an error so they cannot be split independently.


In [ ]:
def allocate_counts(n, ratios):
    if not ratios or any(k not in {"train", "val", "test"} for k in ratios):
        raise ValueError("Use train/val/test ratio keys")
    if any(not math.isfinite(v) or v < 0 for v in ratios.values()) or not math.isclose(sum(ratios.values()), 1.0):
        raise ValueError("Ratios must be finite, nonnegative, and sum to 1")
    active = [k for k, v in ratios.items() if v > 0]
    if n < len(active):
        raise ValueError("{} groups cannot fill {} partitions. Use fewer partitions or a smaller grouping unit.".format(n, len(active)))
    targets = {k: n * ratios[k] for k in active}
    counts = {k: max(1, math.floor(targets[k])) for k in active}
    while sum(counts.values()) > n:
        k = max((k for k in active if counts[k] > 1), key=lambda k: counts[k] - targets[k])
        counts[k] -= 1
    while sum(counts.values()) < n:
        k = max(active, key=lambda k: targets[k] - counts[k])
        counts[k] += 1
    return counts


def assign_splits(pairs, level, ratios, seed):
    if level not in {"well", "batch"}:
        raise ValueError("GROUP_LEVEL must be 'well' or 'batch'")
    groups = defaultdict(list)
    for row in pairs:
        key = row["batch"] if level == "batch" else "/".join([row["batch"], row["video"].upper(), row["well"].upper()])
        groups[key].append(row)
    strata = defaultdict(list)
    for key, rows in sorted(groups.items()):
        conditions = {(r["death_type"], r["cell_type"]) for r in rows}
        if level == "well" and len(conditions) != 1:
            raise ValueError("Well occurs in multiple conditions: " + key)
        stratum = next(iter(conditions)) if level == "well" else ("all batches",)
        strata[stratum].append(key)
    rng = random.Random(seed)
    assignments = {}
    for stratum, keys in sorted(strata.items()):
        counts = allocate_counts(len(keys), ratios)
        rng.shuffle(keys)
        offset = 0
        for split, count in counts.items():
            for key in keys[offset:offset + count]:
                assignments[key] = split
            offset += count
    manifest = [dict(row, group_id=key, split=assignments[key])
                for key, rows in sorted(groups.items()) for row in rows]
    seen = defaultdict(set)
    for row in manifest:
        seen[row["group_id"]].add(row["split"])
    if any(len(splits) != 1 for splits in seen.values()):
        raise ValueError("Biological group leakage detected")
    return manifest

manifest = assign_splits(pairs, GROUP_LEVEL, SPLIT_RATIOS, SEED)
summary = Counter((r["split"], r["death_type"], r["cell_type"]) for r in manifest)
print("split / death type / cell type: paired images (biological groups)")
for key, count in sorted(summary.items()):
    group_count = len({r["group_id"] for r in manifest if (r["split"], r["death_type"], r["cell_type"]) == key})
    print(" / ".join(key), ":", count, "pairs (", group_count, "groups)")
for split in SPLIT_RATIOS:
    rows = [r for r in manifest if r["split"] == split]
    print(split, ":", len(rows), "pairs;", "{:.1%}".format(len(rows) / len(manifest)))


## Copy to a fresh output directory

Run after reviewing the preview. An existing output directory is rejected to prevent stale files from an earlier split. To rerun, choose a new `OUTPUT_ROOT` in the configuration cell. Original images are never moved or modified.

Output uses capitalized `Phase` for compatibility with the annotation notebook:
```
DeathScope_biological_split/
  manifest.csv
  unmatched_files.csv
  split_config.json
  train/Batch1/Apoptosis/MEF_Phase/*.png
  train/Batch1/Apoptosis/MEF_Green/*.png
  val/... and test/...
```


In [ ]:
def export_split(source_root, output_root, manifest, unmatched, config):
    source_root, output_root = Path(source_root).resolve(), Path(output_root).resolve()
    if output_root == source_root or source_root in output_root.parents or output_root in source_root.parents:
        raise ValueError("Output and source must be separate directory trees")
    if output_root.exists():
        raise FileExistsError("Choose a fresh OUTPUT_ROOT: " + str(output_root))
    output_root.mkdir(parents=True)
    exported = []
    for row in manifest:
        out_row = dict(row)
        for channel, suffix in [("phase", "Phase"), ("green", "Green")]:
            relative = Path(row["split"]) / row["batch"] / row["death_type"] / (row["cell_type"] + "_" + suffix) / Path(row[channel]).name
            destination = output_root / relative
            destination.parent.mkdir(parents=True, exist_ok=True)
            if destination.exists():
                raise FileExistsError("Destination collision: " + str(destination))
            shutil.copy2(source_root / row[channel], destination)
            out_row[channel + "_output"] = relative.as_posix()
        exported.append(out_row)
    for filename, rows, fields in [
        ("manifest.csv", exported, list(exported[0])),
        ("unmatched_files.csv", unmatched, ["path", "reason"]),
    ]:
        with (output_root / filename).open("w", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=fields)
            writer.writeheader()
            writer.writerows(rows)
    (output_root / "split_config.json").write_text(json.dumps(dict(config, source_root=str(source_root)), indent=2) + "\n")
    print("Copied", len(exported), "pairs to", output_root)

export_split(SOURCE_ROOT, OUTPUT_ROOT, manifest, unmatched, {
    "group_level": GROUP_LEVEL, "ratios": SPLIT_RATIOS, "seed": SEED,
    "missing_pair_policy": MISSING_PAIR_POLICY,
    "filename_pattern": FILENAME_PATTERN.pattern,
})


## Continue with annotation and augmentation

Process each split/batch/death/cell combination separately. In `annotate_cell_images.ipynb`, replace `dataset = use_dataset(1)` with a configuration like this (choose appropriate preprocessing parameters separately):
```python
dataset = {
    "Image_path": "Data/DeathScope_biological_split/train/Batch1",
    "Death_type": "Apoptosis",
    "Cell_type": "MEF",
}
```
Repeat annotation for validation and test using their own `Image_path`, and for every batch, cell type, and death type present. Annotation writes crops, masks, and labels within that partition's tree. Any background extraction must also preserve the partition.

In `augment_dataset.ipynb`, replace `dataset = use_dataset(1)` with the same configuration for each **training** combination. Leave validation/test crops unaugmented for evaluation. Never pool the partitions or run the random `train_validation_test_split.ipynb` afterward. If collecting files into YOLO `images/{train,val,test}` and `labels/{train,val,test}`, retain these assignments and prefix filenames with batch/death/cell identifiers to prevent collisions. Preserve matching image/label stems. The manifest records the original biological assignment for tracing every derived crop or augmentation.
